<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/3_bert_training/Guided_lab_3_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [BONUS] Guided Lab 3: Named Entity Recognition (NER) with BERT

![](https://uploads5.wikiart.org/images/joseph-kosuth/one-and-three-chairs.jpg!Large.jpg)
<p align="right">
  <i>One and Three Chairs</i>. Joseph Kosuth. (1965). <br>© Joseph Kosuth.
</p>


## Applying BERT for NER in Francophone Historical Newspapers

This notebook is largely inspired and based on [HuggingFace's token classification tutorial](https://huggingface.co/docs/transformers/tasks/token_classification).

The task is applied to historical newspapers in French, borrowed from the [CLEF-HIPE-2020](https://impresso.github.io/CLEF-HIPE-2020/) shared task.

### Objectives

The goal is to train a custom BERT-based NER model on historical newspapers in French.

> *Named Entity Recognition is the process of identifying numerous segments of information referenced in a text and then classifying them into pre-established categories. Entities like ‘‘person’’, ‘‘organization’’, ‘‘region’’, and many more might be considered categories.* [(Jehangir et al., 2023)](https://doi.org/10.1016/j.nlp.2023.100017)


Named Entity Recognition (NER) is a foundational natural language processing task that automatically identifies and categorizes key information elements within unstructured text.
By transforming raw text into structured annotations, NER can enable different types of downstream analyses, including information extraction, text summarization, and knowledge graph construction.

This notebook proposes to tackle this task by adapting a pre-trained transformer model.
Yet, practitioners can also implement NER using various alternative methodologies, ranging from lightweight lexicon- and rule-based systems to production-ready frameworks such as spaCy (see for instance [Melanie Walsh's NER tutorial](https://melaniewalsh.github.io/Intro-Cultural-Analytics/05-Text-Analysis/12-Named-Entity-Recognition.html)).

To learn more about NER in Historical Documents, you can also check the article *Named Entity Recognition and Classification in Historical Documents: A Survey* [(Ehrmann et al., 2023)](https://dl.acm.org/doi/full/10.1145/3604931).

### The data

The data used for this notebook is taken from the [CLEF-HIPE-2020 shared task: *Named Entity Processing on Historical Newspapers*](https://impresso.github.io/CLEF-HIPE-2020/).


The texts to analyse are extracted from historical francophone newspapers spanning the 19th and early 20th centuries (provided via the CLEF-HIPE-2020 campaign). Working with historical press presents several challenges, including OCR (Optical Character Recognition) errors, archaic spellings, irregular layout structures, historical naming conventions and linguistic norms that may differ from modern corpora.

## The task

The goal is to extract named entities based on a pre-defined list of categories. Here, we are using 'coarse' labels, which contain annotations for: persons or named group of people (`pers`); commercial or admistrative organisations (`org`); geographical places at any scale, or electronic contact information (`loc`); newspapers names or doctrines names (`prod`); and date and time (`time`).

The data is annotated using the **BIO-scheme** for NER: as several tokens can be part of the same entity, the first token of this sequence is annotated with `B-<TAG>` (`B` as in 'beginning'), and the following(s) with `I-<TAG>` (`I` as in 'inside'), where `<TAG>` refers to the entity category (here, one of `pers`, `org`, `loc`, `prod`, or `time`). `O` (for 'outside') is the label used for tokens that do not refer to any entity.

Please refer to the [Participation Guidelines](https://zenodo.org/records/3677171) of the original HIPE shared task for more information.

Additional data preprocessing and preparation is implemented in the [pre-processing script on GitHub](https://github.com/d-noe/NLP_DH_F26/blob/main/data/preprocessing/prepro-hipe.py).

## Table of contents

1. [Loading the data](#data_loading)
1. [Loading the pre-trained model](#model_loading)
1. [Data preparation](#prep_data)
2. [Evaluation preparation](#prep_eval)
4. [Training preparation](#prep_training)
5. [Training!](#training)
6. [Predictions, analysis and evaluation](#predictions)

---

First, we will import the different libraries that we will use in this notebook: `numpy` for mathematical calculations (here, mostly for evaluative purposes); `torch` to handle tensors and work on the GPU; `transformes` modules to load pre-trained LMs, handle the data and to train our model; `matplotlib` and `seaborn` for visualisations; and `sklearn`'s `TSNE` for dimensionality reduction.

Additionally, we also fetch the functions from the `helpers`, provided in the class repository, that are implemented to ease the process of loading a `datasets.Dataset` object from the repository data forlder.

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE


from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, AutoTokenizer
from transformers import DataCollatorForTokenClassification

!wget https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github, load_dataset_from_github

### Visualizing Token Representations

Before we dive into the NER task, we also provide helper functions that will be of great help further one. They are implemented to help you inspect and navigate the representation spaces produced by the model. Namely, `get_contextualized_embeddings` and `visualize_token_clusters`.

Indeed, to better understand our model (and the role of the fine-tuning task), it is helpful to visualise how language models represent text. The function below extracts token-wise **contextualized embeddings** from the pre-trained transformer layers and applies t-SNE dimensionality reduction to visualize the representations in 2D. Moreover, the figure shows the NER-tag associated with the representations as a color code, thus enabling to inspect how tokens cluster in vector space based on their NE nature.

*We will use this method a first time to visualise the embeddings produced by the base, pre-trained model (before fine-tuning it for the task), and a second time after the model has been fine-tuned for NER. Keep that in mind, how do you think the representations will compare?*

In [ ]:
#@title Helper functions to visualise tokens embedding distribution based on NER tags (with negative sampes)

def get_contextualized_embeddings(dataset, tokenizer, model, id2label, device="cuda" if torch.cuda.is_available() else "cpu", max_neg_ratio=1.0):
    """
    Extracts contextualized token embeddings and aggregated (BIO-free) label names
    using safe manual batching to prevent tensor shape errors.
    """
    model.to(device)
    model.eval()

    # Get id2label mapping without BIO prefixes (e.g., B-PER -> PER, O -> O)
    agg_id2label = {idx: label.replace('B-', '').replace('I-', '') for idx, label in id2label.items()}

    all_vectors = []
    all_labels = []
    all_positive_indices = []
    all_negative_indices = []
    global_index = 0

    batch_size = 8
    print("Extracting contextual embeddings...")

    for i in tqdm(range(0, len(dataset), batch_size)):
        batch_slice = dataset[i : i + batch_size]

        # Extract input_ids and labels directly from the tokenized dataset slice
        input_ids = batch_slice["input_ids"]
        labels = batch_slice["labels"] if "labels" in batch_slice else batch_slice["ner_tags"]

        # Manually pad inputs and labels for the batch
        max_len = max(len(seq) for seq in input_ids)
        padded_input_ids = [seq + [tokenizer.pad_token_id] * (max_len - len(seq)) for seq in input_ids]
        padded_attention_mask = [[1] * len(seq) + [0] * (max_len - len(seq)) for seq in input_ids]
        padded_labels = [seq + [-100] * (max_len - len(seq)) for seq in labels]

        batch_input_ids = torch.tensor(padded_input_ids, dtype=torch.long).to(device)
        batch_attention_mask = torch.tensor(padded_attention_mask, dtype=torch.long).to(device)
        batch_labels = torch.tensor(padded_labels, dtype=torch.long)

        with torch.no_grad():
            outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, output_hidden_states=True)

        last_hidden_state = outputs.hidden_states[-1].cpu() # (Batch_Size, Seq_Len, 768)

        for b_idx in range(len(input_ids)):
            actual_length = sum(batch_attention_mask[b_idx]).item()
            tokens = tokenizer.convert_ids_to_tokens(batch_input_ids[b_idx][:actual_length])
            raw_labels = batch_labels[b_idx][:actual_length]

            for j, (vector, label_idx) in enumerate(zip(last_hidden_state[b_idx][:actual_length], raw_labels)):
                if label_idx == -100:
                    continue

                agg_label = agg_id2label[label_idx.item()]
                all_vectors.append(vector.numpy())
                all_labels.append(agg_label)

                if agg_label == 'O':
                    all_negative_indices.append(global_index)
                else:
                    all_positive_indices.append(global_index)

                global_index += 1

    # Stratified Sampling for 'O' (Negative) samples
    num_positive = len(all_positive_indices)
    num_negative = len(all_negative_indices)
    num_to_sample_negative = int(num_positive * max_neg_ratio)

    if num_to_sample_negative < num_negative:
        import random
        sampled_negative_indices = random.sample(all_negative_indices, num_to_sample_negative)
    else:
        sampled_negative_indices = all_negative_indices

    all_kept_indices = all_positive_indices + sampled_negative_indices
    final_vectors = [all_vectors[idx] for idx in all_kept_indices]
    final_labels = [all_labels[idx] for idx in all_kept_indices]

    return torch.tensor(final_vectors).numpy(), final_labels

def visualize_token_clusters(vectors, labels, title="Token Embeddings Visualization", perplexity=30):
    """
    Visualizes high-dimensional token vectors in 2D using t-SNE.
    Entities are colored uniquely; 'O' samples are rendered in grey.
    """
    print(f"Reducing {vectors.shape[0]} {vectors.shape[1]}-dim vectors to 2D via t-SNE (perplexity={perplexity})...")
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=1000, random_state=42)
    vectors_2d = tsne.fit_transform(vectors)

    plt.figure(figsize=(8, 8))

    # Identify unique labels
    unique_labels = sorted(list(set(labels)))

    # Define color map: Grey for 'O', categorical colors for others
    # Ensure 'O' is handled specifically
    palette = sns.color_palette("Set2", n_colors=len(unique_labels))
    color_map = {label: (0.7, 0.7, 0.7) if label == 'O' else palette[i] for i, label in enumerate(unique_labels)}

    # Scatter plot, iterating through labels to create a legend
    for label in unique_labels:
        # Filter 2D coordinates for this specific label
        label_indices = np.array([i for i, l in enumerate(labels) if l == label])
        plt.scatter(vectors_2d[label_indices, 0], vectors_2d[label_indices, 1],
                    label=label, color=color_map[label], alpha=0.6, edgecolors='none', s=20)

    plt.title(title, fontsize=16)
    plt.xlabel("t-SNE Dimension 1", fontsize=12)
    plt.ylabel("t-SNE Dimension 2", fontsize=12)
    plt.legend(title="NER Tags", loc="best", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

<a name="data_loading"></a>

## Loading the Data

For the purpose of this notebook, the `.csv` files shared by [CLEF-HIPE-2020](https://github.com/impresso/CLEF-HIPE-2020/tree/master/data/v1.4/fr) have been further prepared (see the [preprocessing script in the class repository](https://github.com/d-noe/NLP_DH_F26/blob/main/data/preprocessing/prepro-hipe.py) for more details) and stored using [arrow format](https://huggingface.co/docs/datasets/about_arrow), to comply with [HuggingFace's `datasets` library](https://huggingface.co/docs/datasets/index).

The dataset contains three data splits: `train`, to train the model; `dev` to tune hyperparameters or assess the model performance before the final evaluation; and `test` to evaluate the model performance on **unseen** data.
The dataset is loaded as a [`DatasetDict` object](https://huggingface.co/docs/datasets/v4.8.4/en/package_reference/main_classes#datasets.DatasetDict), where the first-order keys refer to the data split.

Then, for each split, the dataset contains a list of samples (i.e. annotated text sequences).
These samples are automatically extracted sentences from the raw OCRed newspapers texts.
For each *sample*, the dataset provides a list of `tokens` (pre-tokenized), and the corresponding list of `ner` labels.

Let's inspect the structure of the dataset and examine the label space, noting the use of the **BIO scheme** (`B-` for the beginning of an entity, `I-` for inside, and `O` for outside).

In [ ]:
data_splits =["train","dev","test"]

In [ ]:
dataset = load_dataset_from_github(
    "data/hipe_ner/prepro_hipe_fr", # Path on the GitHub Repository
    splits=data_splits # Names of the data splits to load
)
dataset

In [ ]:
# Get the names of the 'ner' labels (stored as integers in the 'ner' column)
label_list = dataset["train"].features[f"ner"].feature.names

# Build ID -> label and label -> ID equivalence tables
id2label = {
    i: label
    for i, label in enumerate(label_list)
}

label2id = {
    label: i
    for i, label in id2label.items()
}

print("The NER labels in the dataset are:")
for k, v in id2label.items():
    print(f"\tInteger label {k:<3} associated with NER tag: '{v}'")

We can also inspect the content of a given sample:

In [ ]:
# choose a sample ID from the training set
print_ex_id = 98

print(f"{'TOKEN':<15} | {'LABEL ID':<15} | {'NER LABEL':<15}")
print("-" * 50)
for tok, lab in zip(dataset["train"][print_ex_id]["tokens"], dataset["train"][print_ex_id]["ner"]):
    print(f"{tok:<15} | {lab:<15} | {id2label[lab]:<15}")

In [ ]:
# Explore the data
# Compute statistics
# What are the sequences lengths?
# What are the percentage of each NER label?
# Are there many samples without any NE?
# ...
# Feel free to add more cells

<a name="model_loading"></a>
## Loading the Pre-trained Model and Tokenizer

Now that we are familiar with the data, we can tackle the NER task.

For this notebook, we have decided to tackle the task by training (fine-tuning) a pre-trained LM to recognise named entities.

The general idea behind the *pre-train & adapt* paradigm is that language models can first udnergo a general (language-modeling) pre-training, so that the models aquire linguistic knowledge, and then we just have to *adapt* it for our purpose (without the need to start from scratch and to learn both the language and the task at hand).

As we are dealing with French texts, we utilize a French Language Model, here we propose to leverage a distilled variant of CamemBERT, namely [`cmarkea/distilcamembert-base`](https://huggingface.co/cmarkea/distilcamembert-base) [(Delestre & Abibatou, 2022)](https://hal.archives-ouvertes.fr/hal-03674695/file/cap2022.pdf), which has been pre-trained on large-scale French corpora (hence, already *'knows'* French).
(But feel free to experiment with another model!)

So, the first step is to **load the pre-trained model**. The following cells will download the weights of the model and its associated tokenizer and load them in memory.

> 💡 While this task (with a relatively small pre-trained model) is still manageable on CPU, the training step will be longer. If available, connecting to a GPU would make it much faster.


In [ ]:
# are 'GPU's available?
# If using Colab, you can change your runtime to access GPUs
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

In [ ]:
model_name = "cmarkea/distilcamembert-base"

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Load the pre-trained model
# Note: AutoModelForTokenClassification directly appends a classification head to the model
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    device_map=DEVICE,
)


We can check the model's architecture:

In [ ]:
model

In [ ]:
# Feel free to explore the model and tokenizers
# Esp., what kind of tokenization does this tokenizer implement?

In [ ]:
# Inspect the tokenization process
tokenised_example = tokenizer.tokenize(
    dataset["train"][print_ex_id]["tokens"],
    is_split_into_words=True, # the sequence is already a list of 'pre-tokens', not raw text
)

print(f"The original (pre-tokenized) sample contains {len(dataset["train"][print_ex_id]["tokens"])} 'pre-tokens':\n{dataset["train"][print_ex_id]["tokens"]}")
print(f"Using the loaded tokenizer, this yields a sequence of {len(tokenised_example)} tokens for the pre-trained model:\n{tokenised_example}")

<a name="prep_data"></a>

## Data Preparation & Tokenization

Now that we have:
1. Loaded the raw dataset
2. Loaded the pre-trained LM and tokenizer

We must prepare the dataset to comply with our pre-trained model and the training requirements.

### Pre-tokenized data

As we have seen earlier, the raw dataset that we have loaded already contains 'tokens'. Those were in fact obtained from "*simple white space splitting, leaving all punctuation signs (including apostrophes) as separate tokens*", as described in [CLEF-HIPE participation guidelines](https://zenodo.org/records/3677171).
The BIO annotations are directly linked to *this* notion of whitespace splitting tokens, i.e., the labels in the original data are assigned to the *'whitespace splitting tokens'*.

⚠️ Yet, the pre-trained language model uses a different notion of 'tokens'!
Indeed, transformer models use subword tokenization (e.g., BPE), which means that a single *'whitespace splitting token'* (like `"gréviste"`) might be further split into multiple subword tokens (`['▁gré', 'v', 'iste']`).

Thus, while we need to run the pre-tokenized text into the tokenizer for our pre-trained model to be able to process it, we also need to align the (*'whitespace splitting tokens'*-level) labels to the subword-level tokens.

### Tokenization and label alignment

To tokenize our text sequences and align the label IDs with the tokenizer's outputs:
- We use `tokenizer(..., is_split_into_words=True)` to handle pre-tokenized inputs.
- We write an alignment function to ensure that subword tokens generated from the same word correctly inherit or ignore the proper BIO label.
  - The chosen solution to address this issue during data preparation, is to **assign the NER label to the first subword token and map subsequent subword pieces of the same word to a special ignore index** (`-100`) so they are ignored during loss computation.

This is implemented in the `tokenize_and_align_labels` function that we will then apply to our dataset using: `dataset.map(tokenize_and_align_labels, batched=True)` for efficient processing.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

We can now re-inspect the content and labels alignment of our `tokenized_dataset` for a given sample:

In [ ]:
# choose a sample ID from the training set
print_ex_id = 98

# tokenize the pre-tokenized sequence
tokenised_example_ids = tokenizer(
    tokenized_dataset["train"][print_ex_id]["tokens"],
    is_split_into_words=True,
    return_tensors="pt"
)["input_ids"].numpy()[0]

print(f"The length of the tokenized sequence is     : {len(tokenised_example_ids)}")
print(f"The length of the aligned label sequence is : {len(tokenized_dataset["train"][print_ex_id]["labels"])}\n")

print("-" * 75)
print(f"{'TOKEN ID':<15} | {'TOKEN':<15} | {'LABEL ID':<15} | {'NER LABEL':<15}")
print("-" * 70)
for tok_id, lab in zip(
    tokenised_example_ids,
    tokenized_dataset["train"][print_ex_id]["labels"]
):
    # Tokens: convert ID to token
    tok_str = tokenizer.convert_ids_to_tokens([tok_id])[0]
    # Labels: convert label to NER tag if not ignored!
    lab_str = id2label[lab] if not lab == -100 else 'ignored'

    print(f"{str(tok_id):<15} | {tok_str:<15} | {lab:<15} | {lab_str:<15}")

In [ ]:
# Feel free to inspect some more

Finally, we instantiate a [`DataCollator`](https://huggingface.co/docs/transformers/main_classes/data_collator) that will take care of handling (and batching) the data during the training process.

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### 👀 Visualise embeddings *before* training

Alright, so now our data is ready to be processed by the pre-trained LM!

But before diving into training, let's have a look at how the models represent the tokens and structures its semantic space.

>💡 *Remember, those are the representations obtained with a generalist pre-trained LM. Look at the structure of the space. We will re-do this figure after the fine-tuning process, how do you think this will impact the layout/semantic organisation?*

In [ ]:
# Select a representative sample subset of the evaluation/development dataset
# (e.g., take the first 500 examples to keep calculation fast)
sample_size = 500
small_dev_dataset = tokenized_dataset["dev"].shuffle(seed=42).select(range(sample_size))

# 1. Extract Embeddings from the UNTRAINED Model

# Using a max_neg_ratio of 1.0 ensures we include as many 'O' samples as there are entities,
# keeping the visualization balanced.
vectors_before, labels_before = get_contextualized_embeddings(
    dataset=small_dev_dataset,
    tokenizer=tokenizer,
    model=model, # The UNTRAINED model instance
    id2label=id2label,
    max_neg_ratio=1.0, # Sample 1 'O' token for every entity token
)

# 2. Visualize
visualize_token_clusters(vectors_before, labels_before, title="Token Embeddings: BEFORE Fine-Tuning")

<a name="prep_eval"></ad>

## Prepare the evaluation process

The last step before actually training our model for NER, is to define a principled way to measure its performances (so that we can monitor the training procedure and assess its capabilities).

In the context of this notebook, we frame the NER task strictly as a **token classification task**.
As such, evaluation is performed at the individual token level by comparing predicted token IDs against ground-truth token labels. Because subwords and padding tokens (indicated by label IDs of `-100`) must be excluded from scoring, our evaluation logic filters out these special tokens before calculating metrics like accuracy, precision, or recall using NumPy.


**💭 Food for thoughts**

Is evaluating a Named Entity Recognition (NER) model purely at the *token level* sufficient? Does it fit (sufficiently well) the nature of the task?

- If a multi-token entity like *"République française"* is partially recognized (e.g., predicting `B-org` for *"République"* but missing *"française"*), a token-level metric might count it as a partial success. But what would that imply for downstream workflows?
- **Challenge:** Look closely at how token-level accuracy is computed here. If you want to evaluate models based on complete, exact entity spans (calculating span-level Precision, Recall, and F1-scores for chunks like `pers` or `org`), token-level accuracy won't cut it.

*Optional Extension:* Consider replacing or augmenting this token-level evaluation with chunk-level metrics. You can explore standard sequence labeling evaluation libraries like [seqeval](https://github.com/chakki-works/seqeval) to implement span-level evaluation yourself. You may also want to differentiate the evaluation metrics per NER category.


In [ ]:
def compute_metrics(p):
    predictions, labels = p

    # Some models return a tuple of predictions
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to predicted class IDs
    predictions = np.argmax(predictions, axis=-1)

    # Flatten sequences and ignore special/subword tokens marked -100
    mask = labels != -100
    y_true = labels[mask].astype(np.int64)
    y_pred = predictions[mask].astype(np.int64)

    if len(y_true) == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "accuracy": 0.0,
        }

    # Compute the confusion matrix
    n_labels = len(label_list)
    valid = (
        (y_true >= 0) & (y_true < n_labels) &
        (y_pred >= 0) & (y_pred < n_labels)
    )
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    confusion = np.bincount(
        y_true * n_labels + y_pred,
        minlength=n_labels * n_labels,
    ).reshape(n_labels, n_labels)

    # Per-class true positives, false positives, false negatives
    tp = np.diag(confusion).astype(float)
    fp = confusion.sum(axis=0) - tp
    fn = confusion.sum(axis=1) - tp

    precision_per_class = np.divide(
        tp, tp + fp,
        out=np.zeros_like(tp),
        where=(tp + fp) != 0,
    )
    recall_per_class = np.divide(
        tp, tp + fn,
        out=np.zeros_like(tp),
        where=(tp + fn) != 0,
    )
    f1_per_class = np.divide(
        2 * precision_per_class * recall_per_class,
        precision_per_class + recall_per_class,
        out=np.zeros_like(tp),
        where=(precision_per_class + recall_per_class) != 0,
    )

    # Macro-average over classes present in the gold or predicted labels
    active = (tp + fp + fn) > 0

    return {
        "precision": float(precision_per_class[active].mean()),
        "recall": float(recall_per_class[active].mean()),
        "f1": float(f1_per_class[active].mean()),
        "accuracy": float((y_true == y_pred).mean()),
    }


<a name="prep_training"></a>

## Training Preparation: Fine-Tuning Mechanics

### What Happens During Fine-Tuning?
When we fine-tune a pre-trained language model (like `distilcamembert-base`) for token classification, we are performing two complementary operations:
1. **The Classification Head:** A linear classification layer is placed on top of the pre-trained transformer encoder. This head maps the hidden states (e.g., 768-dimensional vectors) to our specific number of output labels (`12` BIO tags). Initially, this head has random weights.
2. **Backpropagation & Adaptation:** During training, gradient descent updates both the newly initialized classification head *and* (optionally) fine-tunes the weights of the underlying transformer layers. This allows the model to adapt its deep linguistic representations—originally trained on modern French—to the spelling variants, layout artifacts, and OCR noise typical of historical newspapers, and directly prepares for the NER task.

### Key Hyperparameters
To control this learning process effectively, we configure several core parameters in `TrainingArguments`, some of those parameters are:
* `learning_rate`: Determines the step size during gradient descent. Usually, pre-trained transformers require very small learning rates (typically between $2\text{e-}5$ and $5\text{e-}5$) to avoid catastrophic forgetting of their pre-trained linguistic knowledge.
* `per_device_train_batch_size`: Defines how many samples are processed simultaneously per GPU/CPU step, balancing memory limits and gradient stability.
* `num_train_epochs`: The number of complete passes over the training dataset.
* `weight_decay`: A regularization technique to prevent overfitting by penalizing large model weights.

In [ ]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir="my_awesome_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # push_to_hub=False, # if you want to publish your trained model on HF (requires API login)
)

In [ ]:
# Instantiate the 'Trainer' that will handle training our model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

<a name="training"></ad>

## 🏃‍➡️ Training!

In [ ]:
# Tested on one run: ~ 3min30 for 5 epochs on GPU (MUCH longer on cpu...!)

trainer.train()

### 👀 Visualise embeddings *after* training

Our model is now fine-tuned for NER!

Let's have a look at how the models represent the tokens and structures its semantic space, now that it has been adapted.

In [ ]:
# Select a representative sample subset of the evaluation/development dataset
# (e.g., take the first 500 examples to keep calculation fast)
sample_size = 500
small_dev_dataset = tokenized_dataset["dev"].shuffle(seed=42).select(range(sample_size))

# 1. Extract Embeddings from the UNTRAINED Model
# Using a max_neg_ratio of 1.0 ensures we include as many 'O' samples as there are entities,
# keeping the visualization balanced.
vectors_before, labels_before = get_contextualized_embeddings(
    dataset=small_dev_dataset,
    tokenizer=tokenizer,
    model=model, # The model has now been TRAINED
    id2label=id2label,
    max_neg_ratio=1.0, # Sample 1 'O' token for every entity token
)

# 2. Visualize
visualize_token_clusters(vectors_before, labels_before, title="Token Embeddings: AFTER Fine-Tuning")

>💡 *Remember how it looked before? How does the layout/semantic organisation differ/resemble?*

<a name="predictions"></ad>

## Predictions and Evaluation

### *Recognising* Named Entities on a single test sample

Let's first get the NER predictions for a chosen *test* sample.

At inference time, the model processes tokenized text sequences to output logits across our label space.

In [ ]:
# Choose the id for the unseen sample ('test' split) you want to test
test_id = 38

test_inputs = tokenizer(
    dataset["test"][test_id]["tokens"], # Tokenize the text to test
    is_split_into_words=True, # It is already split into 'words'
    return_tensors="pt" # We want to get the "pytorch tensors" ("pt") to feed the fine-tuned model directly
).to(DEVICE) # Put everything on the same device

In [ ]:
print(f"Number of tokens in our testing sample : {len(test_inputs['input_ids'][0])}")
# check that the labels have the same length
print(f"Associated number of labels            : {len(tokenized_dataset['test'][test_id]['labels'])}")

In [ ]:
# Run the logits through the model
with torch.no_grad():
  test_logits = model(
      **test_inputs # run the tokenized text through the fine-tuned BERT model for NER
  ).logits # get the output as 'logits' (~ probabilities before softmax)

In [ ]:
# Shape should be: (batch size) x (# tokens) x (# labels)
# --> predictions (over the 12 label classes) per token
print(f"Shape of logits: {test_logits.shape}")

In [ ]:
# We can get the probability distribution over each category for each token
# by using the softmax function:
test_probabilities = torch.nn.functional.softmax(test_logits, dim=2)

In [ ]:
# We can get the predicted labels from the probabilities,
# or directly from the logits!
# (because softmax is purely monotonic — so argmax(logits) = argmax(probs))
predictions = torch.argmax(test_logits, dim=2)

Let's have a look at the predicted labels (and compare them with the 'true' (=annotated) labels)!

In [ ]:
print(f" {'TOKEN':<15} |  {'PREDICTED':<15} |  {'TRUE LABEL':<15}")
print("-" * 55)

for tok, gt, pred in zip(
    tokenizer.convert_ids_to_tokens(test_inputs["input_ids"][0]),
    tokenized_dataset["test"][test_id]["labels"],
    predictions[0].cpu().numpy(),
):
    # Labels: convert label to NER tag if not ignored!
    lab_str_gt = id2label[gt] if not gt == -100 else 'ignored'
    lab_str_pred = id2label[pred] if not pred == -100 else 'ignored'

    print(f" {tok:<15} |  {f'({pred}, {lab_str_pred})':<15} |  {f'({gt}, {lab_str_gt})':<15}")

We can also inspect the probability distribution over each category for each token in a sequence. The following code provides an helper function to do so:

In [ ]:
#@title Token prediction inspection helper

import altair as alt
import pandas as pd

def plot_ner_probabilities(
    tokens,
    probabilities,
    encoding,
    id2label,
    ground_truth=None,
    aggregation="first",
    width=700,
):
    """
    Interactive NER visualization with optional ground truth.

    Parameters
    ----------
    tokens : list[str]
        Original words in the sentence.
    probabilities : torch.Tensor
        Softmax probabilities, shape (1, seq_len, num_labels).
    encoding : tokenizer output
        Hugging Face BatchEncoding with word_ids().
    id2label : dict
        Mapping from label IDs to label names.
    ground_truth : list[str] or list[int], optional
        One true NER label per original word. Labels may be
        strings or integer label IDs.
    aggregation : {"first", "mean"}
        Aggregation method for subword probabilities.
    width : int
        Width of the probability chart.

    Returns
    -------
    alt.VConcatChart
    """
    if aggregation not in ("first", "mean"):
        raise ValueError("aggregation must be 'first' or 'mean'")

    if ground_truth is not None and len(ground_truth) != len(tokens):
        raise ValueError("ground_truth must have one label per word")

    probs = probabilities.detach().cpu().numpy()[0]
    word_ids = encoding.word_ids(batch_index=0)

    # Preserve the model's label-ID order.
    labels = [
        id2label[i] if i in id2label else id2label[str(i)]
        for i in range(len(id2label))
    ]

    def to_label(value):
        if value is None:
            return "Unknown"
        if isinstance(value, int):
            return id2label.get(value, id2label.get(str(value), str(value)))
        return str(value)

    # Aggregate subword probabilities to original words.
    word_probs = []
    for word_idx in range(len(tokens)):
        positions = [
            i for i, wid in enumerate(word_ids)
            if wid == word_idx
        ]
        if not positions:
            raise ValueError(
                f"No subword positions found for word {word_idx}"
            )

        p = (
            probs[positions[0]]
            if aggregation == "first"
            else probs[positions].mean(axis=0)
        )
        word_probs.append(p)

    predicted_labels = [
        labels[int(p.argmax())] for p in word_probs
    ]

    true_labels = (
        [to_label(label) for label in ground_truth]
        if ground_truth is not None
        else [""] * len(tokens)
    )

    # One row per original word.
    token_df = pd.DataFrame({
        "token_index": list(range(len(tokens))),
        "abstract_token": [f"T{i}" for i in range(len(tokens))],
        "token": tokens,
        "predicted_label": predicted_labels,
        "ground_truth": true_labels,
    })

    token_df["summary"] = token_df.apply(
        lambda r: (
            f'Token: "{r["token"]}"  |  Position: {r["token_index"]}  '
            f'|  Predicted: {r["predicted_label"]}'
            + (
                f'  |  Ground truth: {r["ground_truth"]}'
                if ground_truth is not None else ""
            )
        ),
        axis=1,
    )

    prob_df = pd.DataFrame([
        {
            "token_index": i,
            "token": tokens[i],
            "label": labels[j],
            "probability": float(word_probs[i][j]),
        }
        for i in range(len(tokens))
        for j in range(len(labels))
    ])

    selection = alt.selection_point(
        name="selected_token",
        fields=["token_index"],
        on="click",
        clear=False,
        value=[{"token_index": 0}],
    )

    sequence_width = max(width, 42 * len(tokens))

    base = alt.Chart(token_df).encode(
        x=alt.X(
            "token_index:O",
            axis=None,
            title=None,
            scale=alt.Scale(paddingInner=0.08),
        ),
        tooltip=[
            alt.Tooltip("abstract_token:N", title="Token ID"),
            alt.Tooltip("token:N", title="Original word"),
            alt.Tooltip("predicted_label:N", title="Predicted label"),
            alt.Tooltip("ground_truth:N", title="Ground truth"),
        ],
    )

    # Abstract tokens, with a visible selection cue.
    token_boxes = base.mark_rect(
        cornerRadius=4,
        height=34,
        strokeWidth=2,
    ).encode(
        color=alt.condition(
            selection,
            alt.value("#4C78A8"),
            alt.value("#E8EDF2"),
        ),
        stroke=alt.condition(
            selection,
            alt.value("#23466A"),
            alt.value("#FFFFFF"),
        ),
    ).add_params(selection)

    token_text = base.mark_text(
        fontSize=12,
        color="#20252B",
    ).encode(
        text="abstract_token:N",
    )

    token_chart = alt.layer(
        token_boxes, token_text
    ).properties(
        width=sequence_width,
        height=42,
        title="Select a token (hover to inspect)",
    )

    # Ground-truth labels are persistently visible beneath tokens.
    if ground_truth is not None:
        truth_chart = (
            alt.Chart(token_df)
            .mark_text(fontSize=10, color="#A66B00", limit=90)
            .encode(
                x=alt.X("token_index:O", axis=None),
                text=alt.Text("ground_truth:N"),
                tooltip=[
                    alt.Tooltip("token:N", title="Token"),
                    alt.Tooltip("ground_truth:N", title="Ground truth"),
                ],
            )
            .properties(
                width=sequence_width,
                height=24,
            )
        )
        sequence_chart = alt.vconcat(
            token_chart, truth_chart, spacing=0
        )
    else:
        sequence_chart = token_chart

    # A persistent header showing which word is being inspected.
    selected_info = (
        alt.Chart(token_df)
        .transform_filter(selection)
        .mark_text(
            align="left",
            baseline="middle",
            fontSize=13,
            fontWeight="bold",
            color="#303840",
        )
        .encode(text=alt.Text("summary:N"))
        .properties(
            width=sequence_width,
            height=30,
        )
    )

    # Keep the category order fixed, independent of probability.
    probability_chart = (
        alt.Chart(prob_df)
        .transform_filter(selection)
        .mark_bar(
            cornerRadiusEnd=3,
            color="#4C78A8",
        )
        .encode(
            x=alt.X(
                "probability:Q",
                title="Probability",
                scale=alt.Scale(domain=[0, 1]),
                axis=alt.Axis(format=".0%"),
            ),
            y=alt.Y(
                "label:N",
                title="NER category",
                sort=labels,
            ),
            tooltip=[
                alt.Tooltip("token:N", title="Token"),
                alt.Tooltip("label:N", title="Category"),
                alt.Tooltip(
                    "probability:Q",
                    title="Probability",
                    format=".2%",
                ),
            ],
        )
        .properties(
            width=width,
            height=max(150, 25 * len(labels)),
            title="Probability distribution",
        )
    )

    return (
        alt.vconcat(
            sequence_chart,
            selected_info,
            probability_chart,
            spacing=10,
        )
        .configure_view(stroke=None)
        .configure_axis(
            labelFontSize=12,
            titleFontSize=12,
        )
        .configure_title(
            fontSize=14,
            anchor="start",
        )
    )


In [ ]:
# tokens and labels
tokens = dataset["test"][test_id]["tokens"]
true_labels = dataset["test"][test_id]["ner"]
true_labels = [id2label[l] for l in true_labels]

# The model's mapping from class IDs to NER labels.
id2label = model.config.id2label

chart = plot_ner_probabilities(
    tokens=tokens,
    probabilities=test_probabilities,
    encoding=test_inputs,
    id2label=id2label,
    ground_truth=true_labels,
)

chart


### Running evaluation at scale

Then, we might also want to obtain a more significant measure for our trained model's capabilities.  

We can do it by evaluating its performance on the whole test set.

> 💡 Note: We would apply a similar process to extract *Name Entities Tags* at scale on a non-annotated corpus of interest (except for the evaluation procedure, as the goal would be to automatically extract entities for unannotated data).

In [ ]:
# A. GET THE PREDICTIONS ON UNSEEN TEXTS

# Testing samples = only (pre-tokenized) text
# Prepare data for test-time inference
# -> remove original columns such as "tokens" and "ner"
test_dataset = tokenized_dataset["test"].remove_columns(
    [col for col in tokenized_dataset["test"].column_names
     if col not in tokenizer.model_input_names + ["labels"]]
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    collate_fn=data_collator,  # re-use the same datacollator
)

# Set model to 'eval' mode
model.eval()

# Extract the predictions of the model
# + gather 'ground-truth' labels
all_logits = []
all_labels = []

# Run the samples through the NER-adapted LM
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        # Get logits (not argmax predictions) and gold labels
        logits = model(**batch).logits

        all_logits.extend(logits.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

In [ ]:
# Pad sequences to a common length so they form 2D/3D arrays
max_len = max(len(x) for x in all_labels)

all_logits = np.stack([
    np.pad(x, ((0, max_len - len(x)), (0, 0)), constant_values=0)
    for x in all_logits
])

all_labels = np.stack([
    np.pad(x, (0, max_len - len(x)), constant_values=-100)
    for x in all_labels
])

If we did not have access to annotations we would stop here and use the extracted *Name Entity Tags* for downstream analysis.

But, here we are also interested in measuring our model's capabilities on the **unseen test set**. So, let's compute the performance metrics:

In [ ]:
# compute_metrics applies argmax to the logits internally
test_metrics = compute_metrics((all_logits, all_labels))

print("=== Test Set Metrics ===")
for metric, value in test_metrics.items():
  print(f" {metric:<10}: {value:.4f}")

In [ ]:
# [Opt.]
# Alternatively, you can also get evaluation results
# directly by using our previously instantiated `Trainer` object:
test_results = trainer.predict(tokenized_dataset["test"])

# Extract and display the computed metrics
# (e.g., F1, precision, recall, accuracy)
print("=== Test Set Evaluation Metrics ===")
for metric_name, value in test_results.metrics.items():
  print(f"{metric_name}: {value:.4f}")

# Optional: If you want to inspect specific predictions vs true labels
# predictions = test_results.predictions.argmax(axis=2)
# label_ids = test_results.label_ids